# FX Trigger Model for optimal exchange

In [ ]:
from datetime import date
from importlib import reload
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

candidate_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((
    p.resolve() for p in candidate_roots
    if (p / 'src' / 'cbr_loader.py').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Не найден корень проекта с папкой src')
project_root_str = str(PROJECT_ROOT)
# Корень проекта должен находиться первым в sys.path; при повторном запуске
# очищаем cache пакета src, если kernel был запущен из другого checkout.
sys.path = [item for item in sys.path if item != project_root_str]
sys.path.insert(0, project_root_str)
cached_src = sys.modules.get('src')
expected_src = (PROJECT_ROOT / 'src').resolve()
if cached_src is not None:
    cached_file = Path(getattr(cached_src, '__file__', '') or '.').resolve()
    if expected_src not in cached_file.parents:
        for module_name in [name for name in tuple(sys.modules) if name == 'src' or name.startswith('src.')]:
            del sys.modules[module_name]

import src.benchmark as benchmark_module
import src.client_simulation as client_simulation_module
import src.pipeline as pipeline_module
import src.pipeline_visualization as pipeline_viz
import src.timezone_map as timezone_map_module
import src.visualization as policy_viz

from src.cbr_loader import CURRENCIES, load_cbr_history
from src.market_data import build_daily_market_panel
from src.outcomes import add_future_outcomes
from src.features import build_features
from src.config import YuraPipelineConfig
from src.models import ML_FEATURES
from src.rules import RULE_LIBRARY
from src.targets import build_yura_targets
from src.pipeline import run_yura_pipeline
from src.selector import build_opportunity_selector
from src.reporting import compare_backtest_summaries
from src.moex_live import (
    evaluate_signal_relevance, load_current_moex_quotes,
    load_historical_moex_hourly,
    stamp_signals_with_moex_reference,
)

START_DATE = date(2018, 1, 1)
END_DATE = date.today()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'cbr'
# pooled — основной устойчивый режим; hybrid/per_currency — диагностические
# варианты той же архитектуры без изменения контрактов и отчетов.
ML_SCOPE = 'pooled'
CONFIG = YuraPipelineConfig(ml_scope=ML_SCOPE, holdout_start="2025-01-01",)
# Доступно: 'threshold', 'logistic_regression', 'extra_trees'.
# Меняется только selector; engines, policy, holdout и отчеты идентичны.
SELECTOR_TYPE = 'threshold'
SELECTOR = build_opportunity_selector(SELECTOR_TYPE)
CONFIG

## 1. Data, features and labels

In [ ]:
cbr_history = load_cbr_history(
    start_date=START_DATE, end_date=END_DATE, currencies=CURRENCIES, raw_dir=RAW_DIR,
)
rates = (
    cbr_history.pivot(index='available_at', columns='currency', values='normalized_rate')
    .reindex(columns=list(CURRENCIES)).sort_index()
)
rates.columns.name = None
market_panel = build_daily_market_panel(rates)
features = build_features(market_panel)
outcomes = add_future_outcomes(features, horizons=CONFIG.horizons)
dataset, target_registry = build_yura_targets(
    outcomes, horizons=CONFIG.horizons,
    w1_deterioration_bps=CONFIG.w1_deterioration_bps,
    w1_low_percentile=CONFIG.w1_low_percentile,
)
scoring_data = dataset.loc[
    dataset['is_update_day'] & dataset['currency'].isin(CONFIG.currencies)
].copy()

assert not any(name.startswith('target_') for name in ML_FEATURES)
pd.DataFrame({
    'first_available_at': [scoring_data['available_at'].min()],
    'last_available_at': [scoring_data['available_at'].max()],
    'rows': [len(scoring_data)],
    'currencies': [', '.join(CONFIG.currencies)],
})

## 2. Architecture

In [ ]:
architecture = pd.DataFrame([
    {'rule_engine': name, 'parameter_variants': len(variants),
     'features': ', '.join(sorted({feature for item in variants for feature in item.features}))}
    for name, variants in RULE_LIBRARY.items()
]).sort_values('rule_engine').reset_index(drop=True)
architecture

## 3. Walk-forward train and test

In [ ]:
result = run_yura_pipeline(
    scoring_data, target_registry=target_registry, config=CONFIG,
    selector=SELECTOR,
)
display(result.temporal_plan.as_frame())
print(f"Selector: {result.selector_name}")
print(f"Base candidates: {len(result.base_replay.candidates):,}")
print(f"Final holdout signals: {len(result.final_signals):,}")
result.fitted_arbiter

## 4. Audit

In [ ]:
training_audit = result.base_replay.audit.sort_values(
    ['retrain_at', 'target_family', 'horizon', 'engine_type', 'engine_name'],
    na_position='first',
).reset_index(drop=True)
training_audit.tail(30)

## 5. Final test

In [ ]:
signal_backtest_summary = result.backtest_summary
signal_backtest_rows = result.backtest_rows
final_signal_stream = result.final_signals
expected_summary_rows = (
    len(CONFIG.currencies)
    + len(CONFIG.currencies) * len(CONFIG.horizons)
    + len(CONFIG.currencies) * len(CONFIG.horizons) * len(CONFIG.target_families)
)
assert len(signal_backtest_summary) == expected_summary_rows
display(result.action_summary)
display(signal_backtest_summary)
display(result.holdout_coverage)
result.holdout_quarterly_stability

## 6. Diagnostic

In [ ]:
display(result.rule_baseline_summary.sort_values(
    ['currency', 'target_family', 'horizon', 'lift'],
    ascending=[True, True, True, False],
).head(30))
display(result.ml_score_diagnostics.sort_values(
    ['currency', 'target_family', 'horizon']
).head(30))

## 7. Plots

Все графики используют уже рассчитанные объекты — тяжелую ячейку повторять не нужно.

In [ ]:
reload(policy_viz)
reload(pipeline_viz)

policy_viz.plot_policy_currency_overview(signal_backtest_summary)
plt.show()

policy_viz.plot_policy_horizon_heatmaps(
    signal_backtest_summary, target_families=CONFIG.target_families,
)
plt.show()

pipeline_viz.plot_holdout_horizon_mix(final_signal_stream)
plt.show()

pipeline_viz.plot_holdout_sequential_metrics(
    signal_backtest_rows, rolling_signals=25, warmup_signals=20,
)
plt.show()

In [ ]:
# Используются уже рассчитанные scoring_data и final_signal_stream.
reload(policy_viz)

PRESENTATION_CURRENCY = 'AMD'
fig, ax = plt.subplots(figsize=(14, 5))
policy_viz.plot_signal_series(
    scoring_data,
    final_signal_stream,
    currency=PRESENTATION_CURRENCY,
    start_date='2026-01-01',  # result.temporal_plan.holdout_start,
    title=f'RUB/{PRESENTATION_CURRENCY}: сигналы',
    ax=ax,
)
fig.tight_layout()
plt.show()


In [ ]:
quarterly_lift = result.holdout_quarterly_stability.copy()
if quarterly_lift.empty:
    raise ValueError('Квартальный holdout-отчёт пуст')

quarterly_lift['quarter'] = quarterly_lift['quarter'].astype(str)
quarterly_lift['lift'] = pd.to_numeric(quarterly_lift['lift'], errors='coerce')
quarterly_lift = quarterly_lift.sort_values(['currency', 'quarter'])

currencies = [currency for currency in CONFIG.currencies if currency in set(quarterly_lift['currency'])]
fig, axes = plt.subplots(3, 2, figsize=(15, 13), constrained_layout=True)
axes = axes.ravel()

for ax, currency in zip(axes, currencies):
    currency_rows = quarterly_lift.loc[quarterly_lift['currency'].eq(currency)].reset_index(drop=True)
    positions = list(range(len(currency_rows)))

    ax.plot(
        positions, currency_rows['lift'],
        color='#D92D2D', marker='o', markersize=6, linewidth=2.2,
    )
    ax.axhline(1.0, color='#222222', linestyle='--', linewidth=1.2)
    ax.set_xticks(positions, currency_rows['quarter'], rotation=35, ha='right')
    ax.set_title(f'RUB/{currency}', loc='left', fontweight='bold', color='#111111')
    ax.set_ylabel('Квартальный lift', color='#111111')
    ax.set_facecolor('#F7F7F7')
    ax.grid(axis='y', color='#D0D0D0', linewidth=0.8, alpha=0.75)
    ax.tick_params(colors='#222222')

    finite_lift = currency_rows['lift'].dropna()
    upper_limit = max(1.25, float(finite_lift.max()) * 1.18 if not finite_lift.empty else 1.25)
    ax.set_ylim(-0.08, upper_limit)

    for position, row in currency_rows.iterrows():
        if pd.notna(row['lift']):
            ax.annotate(
                f"n={int(row['signal_count'])}",
                (position, row['lift']), xytext=(0, 8), textcoords='offset points',
                ha='center', fontsize=8, color='#222222',
            )

for ax in axes[len(currencies):]:
    ax.remove()

fig.suptitle('Устойчивость lift по кварталам и валютам', fontsize=18, fontweight='bold', color='#111111')
plt.show()


In [ ]:
# Использует уже рассчитанный signal_backtest_rows — pipeline перезапускать не нужно.

FAST_HORIZONS = (1, 3)
SLOW_HORIZONS = (10, 20)

fast_slow_signals = signal_backtest_rows.loc[
    signal_backtest_rows['signal'].fillna(False).astype(bool)
].copy()
fast_slow_signals['horizon'] = pd.to_numeric(
    fast_slow_signals['horizon'], errors='coerce'
)
fast_slow_signals['benefit_bps'] = pd.to_numeric(
    fast_slow_signals['benefit_bps'], errors='coerce'
)
fast_slow_signals['speed'] = np.select(
    [
        fast_slow_signals['horizon'].isin(FAST_HORIZONS),
        fast_slow_signals['horizon'].isin(SLOW_HORIZONS),
    ],
    ['fast', 'slow'],
    default='excluded_h5',
)
fast_slow_signals = fast_slow_signals.loc[
    fast_slow_signals['speed'].isin(['fast', 'slow'])
].dropna(subset=['benefit_bps'])
if fast_slow_signals.empty:
    raise ValueError('В signal_backtest_rows нет финальных fast/slow-сигналов с benefit_bps')

def _summarize_fast_slow(frame, group_columns):
    return (
        frame.groupby([*group_columns, 'speed'], as_index=False)
        .agg(
            signal_count=('benefit_bps', 'size'),
            mean_benefit_bps=('benefit_bps', 'mean'),
            median_benefit_bps=('benefit_bps', 'median'),
            positive_benefit_rate=(
                'benefit_bps', lambda values: float((values > 0).mean())
            ),
        )
    )

fast_slow_bps_by_currency = _summarize_fast_slow(
    fast_slow_signals, ['currency']
)
fast_slow_bps_overall = _summarize_fast_slow(
    fast_slow_signals.assign(currency='Все валюты'), ['currency']
)
fast_slow_bps_summary = pd.concat(
    [fast_slow_bps_by_currency, fast_slow_bps_overall], ignore_index=True
)

currency_order = [
    currency for currency in CONFIG.currencies
    if currency in set(fast_slow_signals['currency'])
] + ['Все валюты']
speed_order = ['fast', 'slow']
mean_matrix = (
    fast_slow_bps_summary.pivot(
        index='currency', columns='speed', values='mean_benefit_bps'
    ).reindex(index=currency_order, columns=speed_order)
)
count_matrix = (
    fast_slow_bps_summary.pivot(
        index='currency', columns='speed', values='signal_count'
    ).reindex(index=currency_order, columns=speed_order)
)

positions = np.arange(len(currency_order))
width = 0.36
fig, ax = plt.subplots(figsize=(13, 6))
bars = []
for offset, speed, label, color in (
    (-width / 2, 'fast', 'Быстрые: h=1/3', '#C62828'),
    ( width / 2, 'slow', 'Длинные: h=10/20', '#171717'),
):
    values = mean_matrix[speed].to_numpy(dtype=float)
    current = ax.bar(
        positions + offset, values, width,
        label=label, color=color, alpha=0.94,
    )
    bars.append((current, speed, values))

ax.axhline(0, color='#777777', linewidth=1)
ax.set_xticks(positions, currency_order)
ax.set_ylabel('Средний реализованный benefit, BPS')
ax.set_title(
    'BPS-ценность быстрых и длинных сигналов',
    loc='left', fontweight='bold', color='#171717',
)
ax.grid(axis='y', color='#D9D9D9', linewidth=0.8)
ax.set_axisbelow(True)
ax.margins(y=0.16)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(frameon=False, ncol=2)

span = np.nanmax(np.abs(mean_matrix.to_numpy(dtype=float)))
padding = max(2.0, 0.025 * span) if np.isfinite(span) else 2.0
for current_bars, speed, values in bars:
    counts = count_matrix[speed].to_numpy(dtype=float)
    for rectangle, value, count in zip(current_bars, values, counts):
        if not np.isfinite(value):
            continue
        vertical = padding if value >= 0 else -padding
        ax.text(
            rectangle.get_x() + rectangle.get_width() / 2,
            value + vertical,
            f'{value:.1f}\nn={int(count)}',
            ha='center', va='bottom' if value >= 0 else 'top',
            fontsize=8, color='#171717',
        )

plt.tight_layout()
plt.show()

display(
    fast_slow_bps_summary.assign(
        speed=lambda frame: frame['speed'].map({
            'fast': 'Быстрые: h=1/3',
            'slow': 'Длинные: h=10/20',
        })
    )[[
        'currency', 'speed', 'signal_count', 'mean_benefit_bps',
        'median_benefit_bps', 'positive_benefit_rate',
    ]].style.format({
        'mean_benefit_bps': '{:.1f}',
        'median_benefit_bps': '{:.1f}',
        'positive_benefit_rate': '{:.1%}',
    }).set_caption('Fast/slow BPS — финальные holdout-сигналы')
)


## 8. MOEX signal update

Для проверки используются **две MOEX-котировки одной и той же бумаги**: первая фиксируется в JSON сигнала при его выпуске, вторая загружается в момент проверки. Курс ЦБ из исследовательского ряда нельзя напрямую использовать как первую цену: у него иной источник, момент фиксации и возможный базис к исполнимой цене MOEX.

In [ ]:
# aiomoex асинхронный; Jupyter поддерживает top-level await.
moex_quotes_at_issue = await load_current_moex_quotes(CONFIG.currencies)
display(moex_quotes_at_issue[[
    'currency', 'secid', 'board', 'bid', 'offer', 'last',
    'buy_price_source', 'buy_price_is_executable',
    'quote_at', 'fetched_at', 'trading_status',
]])

In [ ]:
# Выполнить ОДИН РАЗ непосредственно при выпуске сегодняшних сигналов.
# Исторические holdout-сигналы намеренно не прошиваются сегодняшней ценой.
moscow_today = pd.Timestamp.now(tz='Europe/Moscow').tz_localize(None).normalize()
signal_dates = pd.to_datetime(final_signal_stream['available_at']).dt.normalize()
signals_generated_now = final_signal_stream.loc[signal_dates.eq(moscow_today)].copy()

if signals_generated_now.empty:
    print('Сегодня pipeline не сформировал финальных сигналов; MOEX snapshot загружен.')
    signals_with_market_reference = signals_generated_now.copy()
else:
    signals_with_market_reference = stamp_signals_with_moex_reference(
        signals_generated_now, moex_quotes_at_issue,
        side='BUY_FOREIGN', require_executable=True,
    )
    display(signals_with_market_reference[[
        'event_id', 'currency', 'target_family', 'horizon',
        'confidence', 'expected_bps', 'market_reference_secid',
        'market_reference_price', 'market_reference_quote_at', 'expires_at',
    ]])

In [ ]:
# Production: последний отправленный сигнал + текущая цена -> одно решение.
moex_quotes_now = await load_current_moex_quotes(CONFIG.currencies)
if signals_with_market_reference.empty:
    relevance_decision = None
    print('Нет сегодняшних сигналов для проверки актуальности.')
else:
    last_sent_signal = signals_with_market_reference.sort_values('issued_at').iloc[-1]
    current_quote = moex_quotes_now.loc[
        moex_quotes_now['currency'].eq(last_sent_signal['currency'])
    ].iloc[0]
    relevance_decision = evaluate_signal_relevance(
        last_sent_signal, current_quote,
    )
    display(pd.Series(relevance_decision, name='signal_relevance').to_frame())

# Отдельный загрузчик данных для будущего replay (запускать один раз при подготовке данных):
# moex_hourly = await load_historical_moex_hourly(start='2025-01-01')

## 9. Simulations

Один и тот же замороженный поток сигналов независимо проигрывается для четырёх альтернативных часов выпуска: `09:00`, `12:00`, `15:00` и `18:00` МСК. Это взаимоисключающие продуктовые сценарии, а не четыре отправки одному клиенту. Во всех сценариях используется одно и то же случайное распределение переводов, поэтому разница результатов вызвана временем доставки и проверками актуальности.

Клиенты сначала распределяются между валютными коридорами. В каждом коридоре клиент может совершить не более двух переводов в месяц: один перевод случайно назначается на один из доступных сигналов за 1–14-е числа и один — за 15-е число–конец месяца. Если в половине месяца сигналов нет, перевода по сигналу нет. 

Сигнал исходно сформирован на состоянии рынка в `09:00` МСК. Для сценария `09:00` это уже одобренный pipeline сигнал. Для `12:00`, `15:00` и `18:00` провайдер сначала проверяет его актуальность по цене MOEX на момент предполагаемой отправки и не отправляет потерявший актуальность сигнал. Затем действует локальное окно сна `[21:00, 09:00)`: немедленно доставленный сигнал исполняется после проверки провайдера, а отложенный до 09:00 локального времени повторно проверяется клиентом по новой цене MOEX.


In [ ]:
MOEX_HISTORY_START = pd.Timestamp('2025-01-01')
MOEX_HISTORY_END = pd.to_datetime(final_signal_stream['available_at']).max() + pd.Timedelta(days=2)
moex_hourly = await load_historical_moex_hourly(
    CONFIG.currencies, start=MOEX_HISTORY_START, end=MOEX_HISTORY_END,
)
display(moex_hourly.groupby('currency').agg(
    first_price=('available_at', 'min'),
    last_price=('available_at', 'max'),
    hourly_candles=('close', 'size'),
))

In [ ]:
# Используются уже рассчитанные final_signal_stream, signal_backtest_rows и moex_hourly.
# Основной pipeline и загрузку MOEX повторно запускать не нужно.
reload(client_simulation_module)

ClientSimulationConfig = client_simulation_module.ClientSimulationConfig
SIGNAL_HOURS_MSK = (9, 12, 15, 18)
TIMEZONE_SHARES = {
    "MSK": 0.7611,
    "SAMT": 0.0280,
    "YEKT": 0.0872,
    "OMST": 0.0033,
    "KRAT": 0.0423,
    "IRKT": 0.0179,
    "YAKT": 0.0238,
    "VLAT": 0.0237,
    "MAGT": 0.0079,
    "PETT": 0.0048,
}
CURRENCY_SHARES = None  # None = равные доли клиентов по валютным коридорам
RANDOM_STATE = 42

CLIENT_SIMULATION_CONFIG = ClientSimulationConfig(
    total_clients=10_000,
    average_transfer_rub=22_000.0,
    participation_rate=1.0,
    signal_hour_msk=9,  # sweep заменит его каждым сценарием
    signal_reference_hour_msk=9,
    quiet_start_hour=21,
    quiet_end_hour=9,
    max_market_price_age='2h',
    timezone_shares=TIMEZONE_SHARES,
    currency_shares=CURRENCY_SHARES,
    random_state=RANDOM_STATE,
)

timezone_scenarios = client_simulation_module.simulate_client_signal_hours(
    final_signal_stream, moex_hourly,
    evaluation_rows=signal_backtest_rows,
    signal_hours_msk=SIGNAL_HOURS_MSK,
    config=CLIENT_SIMULATION_CONFIG,
)
# Совместимость: отдельный результат сценария 09:00.
timezone_simulation = timezone_scenarios.scenario_results[9]
timezone_scenario_summary = timezone_scenarios.scenario_summary
timezone_aggregate_summary = timezone_scenarios.aggregate_summary

# Час выпуска × часовая зона; валюты объединяются по фактическим переводам.
timezone_hour_summary = client_simulation_module.aggregate_client_summary(
    timezone_scenario_summary,
    group_columns=('signal_hour_msk', 'timezone'),
)
# Средний результат четырёх альтернативных сценариев по каждой зоне.
timezone_overall_summary = client_simulation_module.aggregate_client_summary(
    timezone_aggregate_summary,
    group_columns=('timezone',),
)

display(timezone_hour_summary[[
    'signal_hour_msk', 'timezone', 'signals_considered', 'signals_sent',
    'signals_delayed', 'rejected_by_provider', 'rejected_after_delivery',
    'potential_client_transactions', 'expected_client_transactions',
    'signal_acceptance_rate', 'lift', 'mean_realized_benefit_pct',
    'mean_client_savings_pct', 'net_savings_per_client_rub',
]].style.format({
    'potential_client_transactions': '{:,.0f}',
    'expected_client_transactions': '{:,.0f}',
    'signal_acceptance_rate': '{:.1%}', 'lift': '{:.2f}',
    'mean_realized_benefit_pct': '{:.3f}%',
    'mean_client_savings_pct': '{:.3f}%',
    'net_savings_per_client_rub': '{:,.0f} ₽',
}).set_caption('Результаты каждого часа выпуска по часовым зонам'))

display(timezone_overall_summary[[
    'timezone', 'signals_considered', 'signals_sent', 'signals_delayed',
    'rejected_by_provider', 'rejected_after_delivery',
    'potential_client_transactions', 'expected_client_transactions',
    'signal_acceptance_rate', 'lift', 'mean_realized_benefit_pct',
    'mean_client_savings_pct', 'net_savings_per_client_rub',
]].style.format({
    'potential_client_transactions': '{:,.0f}',
    'expected_client_transactions': '{:,.0f}',
    'signal_acceptance_rate': '{:.1%}', 'lift': '{:.2f}',
    'mean_realized_benefit_pct': '{:.3f}%',
    'mean_client_savings_pct': '{:.3f}%',
    'net_savings_per_client_rub': '{:,.0f} ₽',
}).set_caption('Итог по четырём альтернативным часам выпуска'))


In [ ]:
# 'ALL' — все валюты; также можно указать 'AMD', 'KGS', 'KZT', 'TJS' или 'UZS'.
reload(timezone_map_module)

MAP_CURRENCY = 'ALL'
RUSSIA_MAP_CACHE = PROJECT_ROOT / 'data' / 'raw' / 'maps' / 'russia_subjects.geojson'

timezone_savings_map = timezone_map_module.plot_russian_timezone_savings_map(
    timezone_aggregate_summary,
    currency=MAP_CURRENCY,
    cache_path=RUSSIA_MAP_CACHE,
)
timezone_savings_map
